# 面试问题：成百上千个 LoRA Adapter 怎样共享一个 Base Model 服务？异构 batching、缓存和隔离如何设计？

**一句话回答**：Base 权重只驻留一份，请求通过不可变 `adapter_id+revision` 选择低秩增量 `ΔW=(α/r)AB`。同一 batch 先做共享 base GEMM，再按 adapter 分组做低秩更新；Adapter 从 CPU/存储分页到 GPU，使用引用计数和 LRU。Registry 必须绑定 base revision、rank、dtype、digest 与 tenant ACL，热更新采用校验—预热—原子发布。

本 Notebook 用 NumPy 手写 LoRA forward、异构 batch 聚合、Adapter Registry、GPU cache、调度、两阶段发布、租户隔离和容量指标。关键路径均配中文注释，不调用 PEFT 或 Serving 框架。


In [ ]:
from collections import OrderedDict
from dataclasses import dataclass
import hashlib,math
import numpy as np

# 小矩阵足以验证共享 base 与异构 adapter 的数值语义。
SEED145=14501; rng145=np.random.default_rng(SEED145)
assert SEED145==14501
assert np.isfinite(rng145.normal())
assert 8//2==4


## 1. LoRA 服务的外部合同是 base + adapter revision

对输入 `X`，输出 `XW + scale·(XA)B`。不同训练代码可能交换 A/B 命名，但 shape 与 scale 必须固化。adapter 只对训练时的 base checkpoint 有意义，不能凭 hidden size 相同跨 base 复用。


In [ ]:
D145,O145,R145=6,5,2; W145=rng145.normal(size=(D145,O145)); A145=rng145.normal(size=(D145,R145)); B145=rng145.normal(size=(R145,O145))
def lora_forward145(X,W,A,B,alpha):
    # 先投影到低秩空间，再映射回输出维；scale 采用 alpha/rank。
    return X@W+(alpha/A.shape[1])*(X@A)@B
X145=rng145.normal(size=(4,D145)); Y145=lora_forward145(X145,W145,A145,B145,4)
assert Y145.shape==(4,O145)
assert np.allclose(lora_forward145(X145,W145,A145*0,B145,4),X145@W145)
assert not np.allclose(Y145,X145@W145)


## 2. 异构 batch 共享 base GEMM，再分组计算 LoRA

若逐请求复制完整模型，显存和 batch 利用率都很差。把 batch 中相同 adapter 的行 gather 到一起，计算 `(XA)B` 后 scatter 回原位置；base `XW` 只算一次。生产系统用定制 kernel 融合不同 rank/adapter，而不是 Python 循环。


In [ ]:
adapters145={"a":(A145,B145,4.),"b":(rng145.normal(size=(D145,1)),rng145.normal(size=(1,O145)),2.)}
ids145=np.array(["a","b","a","b"])
def heterogeneous145(X,W,ids,adapters):
    # 共享 base 输出，按 adapter 索引只写对应行的增量。
    out=X@W
    for aid in sorted(set(ids)):
        rows=np.where(ids==aid)[0]; A,B,alpha=adapters[aid]; out[rows]+=(alpha/A.shape[1])*(X[rows]@A)@B
    return out
hetero145=heterogeneous145(X145,W145,ids145,adapters145); naive145=np.stack([lora_forward145(X145[i:i+1],W145,*adapters145[aid])[0] for i,aid in enumerate(ids145)])
assert np.allclose(hetero145,naive145)
assert hetero145.shape==(4,5)
assert len(set(ids145))==2


## 3. Registry 拒绝 base、rank、shape 或 digest 不一致

Adapter metadata 至少包含 tenant、adapter revision、base revision、rank、alpha、dtype、shape、训练模板与制品 digest。服务发现名称不等于授权；请求 principal 还要通过 ACL。不可变 revision 便于并发请求稳定和回滚。


In [ ]:
@dataclass(frozen=True)
class AdapterMeta145:
    tenant:str; name:str; revision:str; base_revision:str; rank:int; alpha:float; digest:str
    def key(self): return (self.tenant,self.name,self.revision)
def make_meta145(tenant,name,rev,base,A,B,alpha):
    # digest 同时覆盖 A/B 原始字节，防止 registry 指向被替换文件。
    digest=hashlib.sha256(A.tobytes()+B.tobytes()).hexdigest(); return AdapterMeta145(tenant,name,rev,base,A.shape[1],alpha,digest)
meta145=make_meta145("t1","support","v2","base-7",A145,B145,4.)
assert meta145.rank==R145
assert len(meta145.digest)==64
assert meta145.key()==("t1","support","v2")


## 4. GPU Adapter Cache 用 byte quota、引用计数和可回收 LRU

活跃请求 retain adapter，结束后 release；淘汰只能选择 refcount=0 的条目。若所有条目都在使用，准入应排队或失败，不能覆盖正在被 kernel 读取的权重。不同 rank 的真实字节而非条目数决定容量。


In [ ]:
class AdapterCache145:
    def __init__(self,capacity): self.capacity=capacity; self.used=0; self.data=OrderedDict(); self.refs={}
    def put(self,key,size):
        # 先淘汰最老且无引用条目，直到新 adapter 能放入。
        while self.used+size>self.capacity:
            victim=next((k for k in self.data if self.refs[k]==0),None)
            if victim is None: raise RuntimeError("adapter_cache_full")
            self.used-=self.data.pop(victim); self.refs.pop(victim)
        self.data[key]=size; self.refs[key]=0; self.used+=size
    def retain(self,key): self.refs[key]+=1; self.data.move_to_end(key)
    def release(self,key): self.refs[key]-=1
cache145=AdapterCache145(100); cache145.put("a",60); cache145.put("b",30); cache145.retain("a"); cache145.put("c",40)
assert "a" in cache145.data and "b" not in cache145.data
assert cache145.used==100
assert cache145.refs["a"]==1


## 5. 调度器平衡 base batching、adapter locality 与等待公平

合并更多 adapter 可增大 base batch，却增加 adapter load 与异构 kernel 开销；只追热点又会饿死冷 adapter。优先级可结合 deadline、age、adapter 是否驻留、rank 成本和 KV 预算，并给冷请求 aging bonus。


In [ ]:
requests145=[{"id":"r1","adapter":"a","age":1,"resident":1},{"id":"r2","adapter":"cold","age":20,"resident":0},{"id":"r3","adapter":"a","age":2,"resident":1}]
def priority145(r):
    # 驻留奖励提升 locality，age 项保证冷 adapter 最终不会饥饿。
    return 2*r["resident"]+.2*r["age"]
order145=sorted(requests145,key=priority145,reverse=True)
assert order145[0]["id"]=="r2"
assert priority145(requests145[0])<priority145(requests145[1])
assert {r["id"] for r in order145}=={"r1","r2","r3"}


## 6. 热加载采用 staging、数值探针和原子发布

下载到 staging 后校验 digest/base/shape/dtype，在固定 probe 上检查输出有限和阈值，再预热 GPU；最后让新请求指向新 revision。旧请求继续持有旧 revision，待 refcount 清零再回收，失败则不改变 active pointer。


In [ ]:
active145={"support":"v1"}
def publish145(name,revision,meta,A,B,expected_base,probe):
    # 所有校验成功后才执行最后一行的原子指针切换。
    if meta.base_revision!=expected_base or A.shape[1]!=meta.rank or B.shape[0]!=meta.rank: raise ValueError("adapter_incompatible")
    if hashlib.sha256(A.tobytes()+B.tobytes()).hexdigest()!=meta.digest: raise ValueError("digest")
    if not np.all(np.isfinite(probe@A@B)): raise ValueError("nonfinite")
    active145[name]=revision
publish145("support","v2",meta145,A145,B145,"base-7",X145[:1])
assert active145["support"]=="v2"
try: publish145("support","bad",meta145,A145[:,:1],B145,"base-7",X145[:1]); raise AssertionError("bad publish")
except ValueError as e: assert str(e)=="adapter_incompatible"
assert active145["support"]=="v2"


## 7. Adapter、KV Cache 与日志按 tenant 隔离

同名 adapter 在不同 tenant 下是不同 key；principal 必须同时获得 adapter 和 base endpoint 权限。不要把用户输入、adapter 路径或 digest 当授权信息。错误消息和 metrics 也避免泄露其他租户有哪些 adapter 或是否命中缓存。


In [ ]:
acl145={("t1","support","v2"):{"alice"},("t2","support","v2"):{"bob"}}
def authorize145(principal,key):
    # Registry key 已包含 tenant，防止只按名称查找串租户。
    if principal not in acl145.get(key,set()): raise PermissionError("adapter_acl")
    return True
assert authorize145("alice",("t1","support","v2"))
try: authorize145("alice",("t2","support","v2")); raise AssertionError("cross tenant")
except PermissionError as e: assert str(e)=="adapter_acl"
assert ("t1","support","v2")!=("t2","support","v2")


## 8. 评测 adapter 规模、异构度和冷启动分布

指标包括 active adapters、GPU hit/eviction/load latency、base batch size、每批唯一 adapter 数、rank 分布、TTFT/TPOT、吞吐、公平性和逐 adapter 质量。受控矩阵等价只证明语义，Punica/S-LoRA 的性能来自 paging、scheduler 与专用 kernel。


In [ ]:
def adapter_bytes145(din,dout,rank,bytes_=2):
    # LoRA 只存 A/B，两矩阵字节与 rank 线性增长。
    return (din*rank+rank*dout)*bytes_
bytes_r8_145=adapter_bytes145(4096,4096,8); bytes_r64_145=adapter_bytes145(4096,4096,64)
assert bytes_r64_145==8*bytes_r8_145
assert bytes_r8_145<4096*4096*2
assert cache145.used<=cache145.capacity


## 面试总结

完整主线是：**base 与 adapter revision 绑定 → 验证 `XW+(α/r)(XA)B` → base GEMM 共享、按 adapter gather/scatter → registry 固化 shape/digest/tenant → GPU cache byte quota+refcount+LRU → locality 与 age 联合调度 → staging 校验/预热/原子发布 → ACL 隔离 → 冷启动、异构 batch、逐 adapter 质量与 TTFT/TPOT 联合评测**。

延伸阅读：[Punica](https://arxiv.org/abs/2310.18547)、[S-LoRA](https://arxiv.org/abs/2311.03285)、[LoRA](https://arxiv.org/abs/2106.09685)。
